# YOLO11n EXP-002 performance evaluation

This notebook evaluates the frozen EXP-002 checkpoint on the `parcel_damage_v3` validation split. It does not retrain YOLO.

### What YOLO does

YOLO is an object detector. Unlike MobileNet, which gives one label for the whole image, YOLO draws a bounding box around visible damage and assigns one of four classes: `minor_damage`, `compressed`, `hole`, or `wet`.

### What you will get

The notebook calculates overall and per-class precision, recall, mAP50, and mAP50–95. It also creates confusion matrices and precision/recall curves. At the end, all result files are downloaded as a ZIP archive.

EXP-002 was compared and rejected using validation, so its held-out test split remains untouched. The expected reported validation scope is 444 images and 681 annotated instances.

Before starting:

1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. Put the complete project folder in Google Drive. It must include `datasets/processed/parcel_damage_v3` and `runs/EXP-002_yolo11n_ontology_corrected/weights/best.pt`.
3. Update `PROJECT_ROOT` if your Drive folder has a different name.

In [ ]:
# Mount Drive so Colab can read the full dataset and trained checkpoint.
# Colab will display an authorization prompt the first time this cell runs.
from google.colab import drive

drive.mount('/content/drive')

## 1. Locate the complete project

In [ ]:
# Path provides readable and operating-system-safe file paths.
from pathlib import Path
import os

# Change this path if your project has another folder name in Google Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Project_Card_board')

# Stop with a clear message if PROJECT_ROOT does not match your Drive folder.
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Project folder not found: {PROJECT_ROOT}\n'
        'Upload the complete project to Drive or update PROJECT_ROOT.'
    )

# Work from the project root so all relative dataset/model paths are correct.
os.chdir(PROJECT_ROOT)
# Keep temporary Ultralytics settings inside the Colab machine.
os.environ['YOLO_CONFIG_DIR'] = '/content/Ultralytics'
print('Project root:', Path.cwd())

## 2. Install the pinned YOLO dependency

In [ ]:
# Install the same Ultralytics version used by the completed experiment.
# Do not reinstall torch or torchvision: Colab already supplies a compatible
# GPU build of those packages. -q only makes the installation output shorter.
%pip install -q ultralytics==8.4.129

## 3. Check the runtime and required files

In [ ]:
# Print software and GPU information so the run can be understood later.
import sys
import torch
import ultralytics

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# data.yaml tells YOLO where the images/labels are and lists the four classes.
# best.pt contains the trained EXP-002 model weights.
DATA_YAML = Path('datasets/processed/parcel_damage_v3/data.yaml')
YOLO_WEIGHTS = Path('runs/EXP-002_yolo11n_ontology_corrected/weights/best.pt')
# Check for everything needed before starting the slower validation run.
required_paths = [
    DATA_YAML,
    Path('datasets/processed/parcel_damage_v3/valid/images'),
    Path('datasets/processed/parcel_damage_v3/valid/labels'),
    YOLO_WEIGHTS,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required project files:\n- ' + '\n- '.join(missing))

# This experiment originally reported 444 validation images. Printing the
# count helps detect an incomplete Drive upload.
valid_images = [p for p in Path('datasets/processed/parcel_damage_v3/valid/images').iterdir() if p.is_file()]
print('Validation images found:', len(valid_images))
print('Preflight check passed.')

## 4. Evaluate the frozen EXP-002 checkpoint

This uses `split='val'` deliberately. Do not change it to `test` for EXP-002.

Important settings:

- `imgsz=640` resizes inputs to the experiment's evaluation size.
- `batch=16` processes up to 16 images together. Reduce it to 8 if Colab runs out of GPU memory.
- `device=0` means the first GPU; the code automatically uses CPU when no GPU exists.
- `plots=True` creates confusion matrices and metric curves.
- `save_json=True` preserves machine-readable predictions.

In [ ]:
from datetime import datetime, timezone
from ultralytics import YOLO

# Use a unique timestamp so repeated notebook runs do not overwrite results.
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
YOLO_PROJECT = Path('/content/results')
YOLO_RUN_NAME = f'EXP-002_validation_{run_stamp}'
YOLO_PROJECT.mkdir(parents=True, exist_ok=True)

# Load the existing best checkpoint. This does not start training.
model = YOLO(str(YOLO_WEIGHTS), task='detect')

# Compare model detections with the known validation labels.
metrics = model.val(
    data=str(DATA_YAML.resolve()),
    split='val',
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project=str(YOLO_PROJECT),
    name=YOLO_RUN_NAME,
    exist_ok=False,
    plots=True,
    save_json=True,
    verbose=True,
)
YOLO_OUTPUT = Path(metrics.save_dir)
print('Results saved to:', YOLO_OUTPUT)

## 5. Display overall and per-class metrics

Metric guide:

- **Precision:** when YOLO reports damage, how often that detection is correct.
- **Recall:** how much of the labeled damage YOLO finds.
- **mAP50:** mean average precision when a predicted box needs at least 50% overlap with the true box.
- **mAP50–95:** a stricter average over overlap thresholds from 50% through 95%.

Values closer to 1 are better, but these metrics describe detection quality—not physical damage severity.

In [ ]:
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

# results_dict contains the four main overall detection metrics.
overall = {key: float(value) for key, value in metrics.results_dict.items()}
display(pd.DataFrame(overall.items(), columns=['Metric', 'Value']))

# Build a second table so each damage class can be examined separately.
class_names = model.names
class_rows = []
for result_index, class_id in enumerate(metrics.box.ap_class_index.astype(int).tolist()):
    precision, recall, map50, map50_95 = metrics.box.class_result(result_index)
    class_rows.append({
        'class': class_names[class_id],
        'precision': float(precision),
        'recall': float(recall),
        'mAP50': float(map50),
        'mAP50-95': float(map50_95),
    })
class_frame = pd.DataFrame(class_rows)
display(class_frame)

# Save the same tables as JSON for later use in a report or presentation.
summary_path = YOLO_OUTPUT / 'notebook_metrics.json'
summary_path.write_text(json.dumps({'overall': overall, 'per_class': class_rows}, indent=2))
print('Saved metric summary:', summary_path)

## 6. Display generated plots

The confusion matrix shows which damage classes are confused. The other curves show how precision, recall, and F1 change as the detection-confidence threshold changes.

In [ ]:
# Ultralytics writes these images automatically when plots=True.
plot_names = [
    'confusion_matrix_normalized.png',
    'confusion_matrix.png',
    'PR_curve.png',
    'P_curve.png',
    'R_curve.png',
    'F1_curve.png',
]
for filename in plot_names:
    path = YOLO_OUTPUT / filename
    if path.exists():
        print(filename)
        display(DisplayImage(filename=str(path)))

## 7. Download all YOLO results

In [ ]:
# Compress all validation tables, plots, and prediction files into one ZIP.
import shutil
from google.colab import files

archive = shutil.make_archive(str(YOLO_OUTPUT), 'zip', root_dir=YOLO_OUTPUT)
print('Created:', archive)
files.download(archive)